# PaliGemma-3B · QLoRA · RISC Captioning

End-to-end notebook for parameter-efficient fine-tuning of PaliGemma-3B (SigLIP + Gemma-2B) on the RISC remote-sensing image captioning dataset using QLoRA.

- **Course:** DI 725 — Transformers and Attention-Based Deep Networks (METU)
- **Author:** Süleyman Ceran
- **Repo:** https://github.com/sceran/paligemma3b-QLoRA-RISC-captioning

Sections:
1. Setup & Configuration
2. Data — RISC Loading, EDA, Image-Level Split
3. Baseline — PaliGemma Zero-Shot Evaluation
4. QLoRA Fine-Tuning
5. Evaluation — Fine-Tuned Model
6. Originality Experiments
7. Final Comparison & Discussion
8. (Optional) Stretch Goals

## 1. Setup & Configuration

Imports, central configuration dictionary, random seeds, and authentication for HuggingFace Hub (gated PaliGemma weights) and Weights & Biases (experiment tracking).

In [ ]:
%pip install -q -U transformers datasets peft bitsandbytes accelerate wandb pycocoevalcap nltk Pillow

In [ ]:
import os
import random
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoProcessor,
    PaliGemmaForConditionalGeneration,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import wandb

Central configuration. Every hyperparameter that affects an experiment lives here — changing values in this cell propagates everywhere.

In [ ]:
CONFIG = {
    "seed": 42,
    "model_id": "google/paligemma-3b-pt-224",
    "dataset_id": "caglarmert/full_riscm",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "dtype": torch.bfloat16,

    "prompt": "caption en",
    "caption_strategy": "random",

    "split": {
        "test_size": 0.10,
        "val_size": 0.05,
    },

    "bnb": {
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_use_double_quant": True,
    },

    "lora": {
        "r": 16,
        "alpha": 32,
        "dropout": 0.05,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        "use_rslora": True,
        "use_dora": False,
        "lora_plus_lr_ratio": 16,
    },

    "freeze_vision_tower": True,
    "freeze_projector": True,

    "training": {
        "num_train_epochs": 2,
        "per_device_train_batch_size": 4,
        "gradient_accumulation_steps": 4,
        "learning_rate": 1e-4,
        "warmup_ratio": 0.03,
        "weight_decay": 1e-6,
        "logging_steps": 25,
        "eval_steps": 500,
        "save_steps": 500,
        "save_total_limit": 2,
        "output_dir": "checkpoints/paligemma_qlora_risc",
    },

    "eval": {
        "num_qualitative": 8,
        "max_new_tokens": 64,
        "num_beams": 1,
    },

    "wandb": {
        "project": "paligemma3b-QLoRA-RISC",
        "entity": None,
        "run_name": "qlora-r16-lr1e4",
    },
}

RESULTS = {}

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])

### Authentication

PaliGemma is a gated model on the Hub — the HF account must have accepted the license at https://huggingface.co/google/paligemma-3b-pt-224. The wandb call opens a browser tab or accepts a key from stdin.

In [ ]:
from huggingface_hub import login as hf_login

hf_login()
wandb.login()

## 2. Data — RISC Loading, EDA, and Image-Level Split

RISC (`caglarmert/full_riscm`): 44,521 satellite images at 224×224, 5 reference captions per image (222,605 total).

Splits are **image-level**: captions of the same image must never appear in different splits.

In [ ]:
ds = load_dataset(CONFIG["dataset_id"])
ds

### Schema Inspection

Inspect the first row to confirm the caption field name (likely `captions` as a list, or `caption_1` through `caption_5`) and adapt the rest of the notebook accordingly.

In [ ]:
first = ds["train"][0]
print("columns:", ds["train"].column_names)
print("image type:", type(first["image"]))
{k: v for k, v in first.items() if k != "image"}

### EDA — Sample Images and Caption-Length Distribution

In [ ]:
def collect_captions(example):
    if "captions" in example:
        return list(example["captions"])
    caption_cols = [c for c in example.keys() if c.startswith("caption")]
    return [example[c] for c in sorted(caption_cols)]

lengths = []
for ex in ds["train"].select(range(2000)):
    for cap in collect_captions(ex):
        lengths.append(len(cap.split()))

plt.figure(figsize=(6, 3))
plt.hist(lengths, bins=40)
plt.xlabel("caption length (words)")
plt.ylabel("count")
plt.title("Caption length distribution (first 2k examples)")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, ex in zip(axes.flat, ds["train"].select(range(6))):
    ax.imshow(ex["image"])
    ax.axis("off")
    ax.set_title(collect_captions(ex)[0][:60], fontsize=8)
plt.tight_layout()
plt.show()

### Image-Level Train / Val / Test Split

Hash a stable image identifier into a deterministic 80 / 5 / 15 split (or whatever ratio CONFIG specifies). Holds across reruns because the seed is fixed.

If each row of `full_riscm` is already an image with all 5 captions grouped, a simple shuffle-and-slice over rows is correct. If captions are stored one-per-row, group first by image id.

In [ ]:
def make_image_level_splits(dataset, test_size, val_size, seed):
    shuffled = dataset.shuffle(seed=seed)
    n = len(shuffled)
    n_test = int(n * test_size)
    n_val = int(n * val_size)
    n_train = n - n_test - n_val
    return DatasetDict({
        "train": shuffled.select(range(n_train)),
        "val": shuffled.select(range(n_train, n_train + n_val)),
        "test": shuffled.select(range(n_train + n_val, n)),
    })

splits = make_image_level_splits(
    ds["train"],
    test_size=CONFIG["split"]["test_size"],
    val_size=CONFIG["split"]["val_size"],
    seed=CONFIG["seed"],
)
{k: len(v) for k, v in splits.items()}

### Caption Selection Strategy

RISC ships 5 captions per image. Default behaviour: sample one per training step. Other rules are evaluated as the Section 6b ablation.

In [ ]:
def select_caption(captions, strategy):
    if strategy == "random":
        return random.choice(captions)
    if strategy == "first":
        return captions[0]
    if strategy == "longest":
        return max(captions, key=len)
    if strategy == "concat":
        return " ".join(captions)
    raise ValueError(strategy)

## 3. Baseline — PaliGemma Zero-Shot Evaluation

Establish the comparison floor before any fine-tuning. Generates captions on the held-out test split, computes the full metric suite, and logs qualitative examples for Section 7's report figure.

In [ ]:
processor = AutoProcessor.from_pretrained(CONFIG["model_id"])
baseline_model = PaliGemmaForConditionalGeneration.from_pretrained(
    CONFIG["model_id"],
    torch_dtype=CONFIG["dtype"],
).to(CONFIG["device"])
baseline_model.eval();

In [ ]:
@torch.no_grad()
def generate_caption(model, processor, image, prompt, max_new_tokens, num_beams):
    inputs = processor(
        text=prompt,
        images=image.convert("RGB"),
        return_tensors="pt",
    ).to(model.device, dtype=CONFIG["dtype"])
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=num_beams,
        do_sample=False,
    )
    text = processor.decode(output_ids[0], skip_special_tokens=True)
    return text[len(prompt):].strip() if text.startswith(prompt) else text.strip()

### Metric Suite

Primary: **CIDEr** (standard for image captioning).
Classic: BLEU-1, BLEU-4, METEOR, ROUGE-L, SPICE.
Modern: CLIPScore, RefPAC-S++.

All metrics use the 5 reference captions per image.

In [ ]:
def compute_caption_metrics(predictions, references):
    """predictions: list[str] of length N. references: list[list[str]] of length N, each inner list has 5 captions."""
    raise NotImplementedError("wire pycocoevalcap (BLEU/METEOR/ROUGE/CIDEr/SPICE) + CLIPScore + RefPAC-S++")

In [ ]:
run = wandb.init(
    project=CONFIG["wandb"]["project"],
    name="baseline-zeroshot",
    config=CONFIG,
    job_type="baseline",
)

In [ ]:
baseline_preds, baseline_refs = [], []
for ex in splits["test"]:
    pred = generate_caption(
        baseline_model,
        processor,
        ex["image"],
        CONFIG["prompt"],
        CONFIG["eval"]["max_new_tokens"],
        CONFIG["eval"]["num_beams"],
    )
    baseline_preds.append(pred)
    baseline_refs.append(collect_captions(ex))

RESULTS["baseline"] = compute_caption_metrics(baseline_preds, baseline_refs)
wandb.log({"baseline/" + k: v for k, v in RESULTS["baseline"].items()})

### Qualitative Examples — Baseline

Log 8 sample images with their ground-truth captions and the zero-shot baseline prediction. Same indices are reused in Sections 5–7 for direct side-by-side comparison.

In [ ]:
qualitative_indices = list(range(CONFIG["eval"]["num_qualitative"]))
RESULTS["qualitative_indices"] = qualitative_indices

table = wandb.Table(columns=["idx", "image", "ref_1", "ref_2", "ref_3", "ref_4", "ref_5", "baseline"])
for i in qualitative_indices:
    ex = splits["test"][i]
    refs = collect_captions(ex)
    pred = baseline_preds[i]
    table.add_data(i, wandb.Image(ex["image"]), *refs, pred)
wandb.log({"qualitative/baseline": table})
wandb.finish()

## 4. QLoRA Fine-Tuning

4-bit NF4 quantization of the base weights + LoRA adapters on the language-model attention and MLP projections. Vision tower and multimodal projector are frozen by default — the standard RS-VLM pattern; Section 6a revisits this choice.

LoRA is configured with **rsLoRA scaling** (α/√r instead of α/r, stable at higher ranks) and **LoRA+** (higher learning rate for the B matrices than the A matrices, no extra parameters).

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=CONFIG["bnb"]["load_in_4bit"],
    bnb_4bit_quant_type=CONFIG["bnb"]["bnb_4bit_quant_type"],
    bnb_4bit_use_double_quant=CONFIG["bnb"]["bnb_4bit_use_double_quant"],
    bnb_4bit_compute_dtype=CONFIG["dtype"],
)

lora_config = LoraConfig(
    r=CONFIG["lora"]["r"],
    lora_alpha=CONFIG["lora"]["alpha"],
    lora_dropout=CONFIG["lora"]["dropout"],
    target_modules=CONFIG["lora"]["target_modules"],
    task_type="CAUSAL_LM",
    use_rslora=CONFIG["lora"]["use_rslora"],
    use_dora=CONFIG["lora"]["use_dora"],
)

In [ ]:
model = PaliGemmaForConditionalGeneration.from_pretrained(
    CONFIG["model_id"],
    quantization_config=bnb_config,
    device_map={"": 0},
)
model = prepare_model_for_kbit_training(model)

if CONFIG["freeze_vision_tower"]:
    for p in model.vision_tower.parameters():
        p.requires_grad = False
if CONFIG["freeze_projector"]:
    for p in model.multi_modal_projector.parameters():
        p.requires_grad = False

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

### Collator

Builds each batch: caption-strategy-selected target text, standard PaliGemma captioning prompt, image, processed by `AutoProcessor` with right-padding. The processor handles the `<image>` / `<bos>` token insertion automatically.

In [ ]:
def make_collate_fn(processor, strategy, prompt, device, dtype):
    def collate(examples):
        prompts = [prompt] * len(examples)
        targets = [select_caption(collect_captions(ex), strategy) for ex in examples]
        images = [ex["image"].convert("RGB") for ex in examples]
        tokens = processor(
            text=prompts,
            images=images,
            suffix=targets,
            return_tensors="pt",
            padding="longest",
        )
        return tokens.to(dtype).to(device)
    return collate

collate_fn = make_collate_fn(
    processor,
    CONFIG["caption_strategy"],
    CONFIG["prompt"],
    CONFIG["device"],
    CONFIG["dtype"],
)

### LoRA+ Parameter Groups

Builds optimizer parameter groups for LoRA+: the B matrices receive a higher learning rate than the A matrices (ratio set in `CONFIG["lora"]["lora_plus_lr_ratio"]`). No extra trainable parameters.

In [ ]:
def build_lora_plus_param_groups(model, base_lr, lora_plus_ratio):
    A_params, B_params, other_params = [], [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "lora_A" in name:
            A_params.append(param)
        elif "lora_B" in name:
            B_params.append(param)
        else:
            other_params.append(param)
    return [
        {"params": A_params, "lr": base_lr},
        {"params": B_params, "lr": base_lr * lora_plus_ratio},
        {"params": other_params, "lr": base_lr},
    ]

### Training

HuggingFace `Trainer` with wandb reporting, gradient checkpointing, and periodic evaluation on the validation slice. The optimizer is constructed manually so the LoRA+ parameter groups take effect.

In [ ]:
training_args = TrainingArguments(
    output_dir=CONFIG["training"]["output_dir"],
    num_train_epochs=CONFIG["training"]["num_train_epochs"],
    per_device_train_batch_size=CONFIG["training"]["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["training"]["gradient_accumulation_steps"],
    learning_rate=CONFIG["training"]["learning_rate"],
    warmup_ratio=CONFIG["training"]["warmup_ratio"],
    weight_decay=CONFIG["training"]["weight_decay"],
    logging_steps=CONFIG["training"]["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["training"]["eval_steps"],
    save_strategy="steps",
    save_steps=CONFIG["training"]["save_steps"],
    save_total_limit=CONFIG["training"]["save_total_limit"],
    bf16=True,
    gradient_checkpointing=True,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    report_to="wandb",
    run_name=CONFIG["wandb"]["run_name"],
    seed=CONFIG["seed"],
)

param_groups = build_lora_plus_param_groups(
    model,
    base_lr=CONFIG["training"]["learning_rate"],
    lora_plus_ratio=CONFIG["lora"]["lora_plus_lr_ratio"],
)
optimizer = torch.optim.AdamW(param_groups, weight_decay=CONFIG["training"]["weight_decay"])

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=splits["train"],
    eval_dataset=splits["val"],
    data_collator=collate_fn,
    optimizers=(optimizer, None),
)

trainer.train()

## 5. Evaluation — Fine-Tuned Model

Generate captions on the same test split with the fine-tuned model. Reuses the qualitative indices from Section 3 so the Section 7 figure shows a clean baseline→fine-tuned diff for the report.

In [ ]:
ft_preds, ft_refs = [], []
for ex in splits["test"]:
    pred = generate_caption(
        trainer.model,
        processor,
        ex["image"],
        CONFIG["prompt"],
        CONFIG["eval"]["max_new_tokens"],
        CONFIG["eval"]["num_beams"],
    )
    ft_preds.append(pred)
    ft_refs.append(collect_captions(ex))

RESULTS["finetuned"] = compute_caption_metrics(ft_preds, ft_refs)
wandb.log({"finetuned/" + k: v for k, v in RESULTS["finetuned"].items()})

In [ ]:
table = wandb.Table(columns=["idx", "image", "reference", "baseline", "finetuned"])
for i in RESULTS["qualitative_indices"]:
    ex = splits["test"][i]
    refs = collect_captions(ex)
    table.add_data(i, wandb.Image(ex["image"]), refs[0], baseline_preds[i], ft_preds[i])
wandb.log({"qualitative/baseline_vs_finetuned": table})

## 6. Originality Experiments

Two experiments designed to push beyond the lab-provided recipe:

- **6a.** Adapt the SigLIP vision tower with LoRA — motivated by satellite imagery being out-of-distribution for the contrastively-trained encoder.
- **6b.** Compare caption-selection strategies — RISC ships 5 captions per image; which selection rule produces the strongest training signal?

### 6a. Vision-Tower LoRA — 2×2 Grid

Holds the Section 4 LM-side LoRA configuration fixed and varies what happens upstream:

| | projector frozen | projector trained |
|---|---|---|
| **vision frozen** | A — RS-VLM convention (this is the Section 4 run) | B |
| **vision LoRA** | C | D — most parameters adapted |

All four cells use the same data, compute budget, seed, and LM-side LoRA settings.

In [ ]:
vision_tower_experiments = [
    {"name": "A_vision-frozen_proj-frozen",  "vision": "frozen", "projector": "frozen"},
    {"name": "B_vision-frozen_proj-trained", "vision": "frozen", "projector": "trained"},
    {"name": "C_vision-lora_proj-frozen",    "vision": "lora",   "projector": "frozen"},
    {"name": "D_vision-lora_proj-trained",   "vision": "lora",   "projector": "trained"},
]

def run_vision_tower_variant(spec):
    raise NotImplementedError("factor Section 4 setup into a function and parameterize over spec")

RESULTS["vision_tower_2x2"] = {spec["name"]: run_vision_tower_variant(spec) for spec in vision_tower_experiments}

### 6b. Caption Selection Strategy Ablation

Four configurations sharing the Section 4 model configuration:

- **random** — sample one of 5 captions per training step (default)
- **first** — always the first caption in the list
- **longest** — caption with the most tokens (most detailed reference)
- **concat** — all 5 captions concatenated into a single target sequence

In [ ]:
caption_strategy_experiments = ["random", "first", "longest", "concat"]

def run_caption_strategy_variant(strategy):
    raise NotImplementedError("reuse Section 4 training loop with CONFIG['caption_strategy']=strategy")

RESULTS["caption_strategy"] = {s: run_caption_strategy_variant(s) for s in caption_strategy_experiments}

## 7. Final Comparison & Discussion

Single consolidated metric table across all runs plus the final qualitative grid for the IEEE report (5+ image rows; each row shows ground truth, baseline, fine-tuned, and the best originality-experiment variant).

In [ ]:
summary_rows = []
summary_rows.append({"run": "baseline", **RESULTS["baseline"]})
summary_rows.append({"run": "finetuned", **RESULTS["finetuned"]})
for name, metrics in RESULTS.get("vision_tower_2x2", {}).items():
    summary_rows.append({"run": f"v6a/{name}", **metrics})
for strategy, metrics in RESULTS.get("caption_strategy", {}).items():
    summary_rows.append({"run": f"v6b/{strategy}", **metrics})

summary_df = pd.DataFrame(summary_rows)
summary_df

In [ ]:
summary_df.to_csv("results_summary.csv", index=False)
with open("results_summary.json", "w") as f:
    json.dump(summary_df.to_dict(orient="records"), f, indent=2)

## 8. (Optional) Stretch Goals

Only attempted if Sections 4–7 conclude with time remaining.

- **PaliGemma 2 side-experiment.** Swap `CONFIG["model_id"]` to `google/paligemma2-3b-pt-224` and rerun the main QLoRA configuration. Reports the v1→v2 gap on an identical setup.
- **LLM-as-judge.** Sample 100–200 test images, prompt an LLM to rank baseline vs fine-tuned captions on relevance and specificity, report inter-judge agreement with classic metrics.